# 🎮 AI Cyber Academy: Autonomous Vulnerability Discovery
### Level 1: The Code Doctor (Finding & Healing Broken Locks)

Welcome to the **AI Cyber Academy**! In this hands-on security engineering lab, we explore automated vulnerability discovery and remediation using the modern Google GenAI SDK (`google-genai`) and frontier reasoning models.

> 🧭 **Dual-Layer Learning Guide:**
> * 🧒 **For Curious Beginners (and 12-year-old future hackers):** Read the 🕵️‍♂️ **Mission Briefings** and 💡 **Plain-English Stories** to learn the riddles behind how computer locks break.
> * 🛡️ **For DevSecOps & Security Engineers:** Dive into the formal CWE mappings, Pydantic schemas, AST diffs, and enterprise CI/CD verification harnesses.

---

## Environment Setup

We use `uv` for fast, reproducible virtual environment and package management. In your terminal, run:

```bash
# Synchronize project dependencies
uv sync

# Configure your Gemini API key in .env
cp .env.example .env

# Launch the Jupyter Notebook environment
uv run jupyter notebook
```

---

## Technical Overview & Benchmarks

Gemini reasoning models replace pattern-matching static analysis with contextual, multi-step agentic reasoning.

### Performance Metrics

| Metric / Benchmark | Gemini 3.8 Flash Cyber | Industry Standard / Competitors |
|:---|:---|:---|
| **Vulnerability Discovery** | >70% Success (20 languages) | Variable depending on the model |
| **CyberGym Score** | 86.2% | ~83.8% (Anthropic Opus 5) |
| **CWE-Bench Patching** | 47.2% | Limited autonomous capabilities |
| **Chrome Bug Remediation** | 2.6x more correct patches | Standard large language models |
| **Access Distribution** | Restricted (Fairwind Program) / Public API | Public API / Commercial access |

---

### Gemini vs. Traditional Cybersecurity Tools

AI models do not replace vulnerability scanners, SIEMs, or human researchers—they add an autonomous reasoning layer to existing security pipelines:

| Traditional Tools | AI-Assisted Security |
|:---|:---|:---|
| Detect suspicious syntax patterns | Assess dynamic exploitability & impact |
| Report raw vulnerability alerts | Investigate root cause within application context |
| Generate static finding logs | Propose verified candidate patches |
| Require manual patch authoring | Generate complete, safe replacement code |
| Require manual validation | AI-assisted testing and verification |
| Periodic scheduled scanning | Continuous agentic analysis |


## 1. Initialization & Dynamic Model Selector

We initialize `genai.Client()` with automatic credentials resolution from `.env`.

Use the interactive widget below to dynamically select your target model. By default, it uses **`gemini-3.8-flash`** for high-speed agentic reasoning, with **`gemini-2.5-flash`** available as a stable production alternative.

In [1]:
import os
import difflib
import sqlite3
import ipywidgets as widgets
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from pydantic import BaseModel, Field
from IPython.display import display, Markdown, HTML, clear_output

# Load API credentials from local .env file
load_dotenv(find_dotenv())
client = genai.Client()

# Dynamic Model Selector: Defaults to gemini-3.8-flash
AVAILABLE_MODELS = [
    ("Gemini 3.8 Flash (Default - High Speed & Cyber Reasoning)", "gemini-3.8-flash"),
    ("Gemini 2.5 Flash (High Availability Production Fallback)", "gemini-2.5-flash"),
    ("Gemini 2.5 Pro (Deep Multi-Step Reasoning)", "gemini-2.5-pro"),
]

model_dropdown = widgets.Dropdown(
    options=AVAILABLE_MODELS,
    value="gemini-3.8-flash",
    description="Target Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px")
)

display(Markdown("### ⚙️ Client Initialized & Dynamic Model Selector"))
display(Markdown("Select the target model for the vulnerability audit (defaults to **`gemini-3.8-flash`**):"))
display(model_dropdown)


### ⚙️ Client Initialized & Dynamic Model Selector

Select the target model for the vulnerability audit (defaults to **`gemini-3.8-flash`**):

Dropdown(description='Target Model:', layout=Layout(width='620px'), options=(('Gemini 3.8 Flash (Default - Hig…

## 2. Defining Target Code with a Vulnerability

> 🕵️‍♂️ **Mission Briefing: The Broken Lock (SQL Injection)**
> * **The Story:** Imagine a digital filing cabinet. The guard asks for your username. Instead of safely looking up your folder, the guard takes whatever word you speak, glues it directly into the database command, and executes it!
> * **The Hacker's Riddle:** What if an attacker types: `' OR '1'='1`? The guard reads: *"Find user admin OR check if 1 equals 1"*. Since 1 always equals 1, the door swings wide open and dumps every single secret in the building!

Below is the vulnerable Python function `get_user_data`. Notice line 7 where `f"...{username}"` glues user input directly into SQL:

In [2]:
# Target code containing a critical security flaw
vulnerable_code = """import sqlite3

def get_user_data(username: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    # VULNERABILITY: Unsanitized user input directly interpolated into SQL query
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    result = cursor.fetchall()
    conn.close()
    return result
"""

display(Markdown("### 🎯 Target Code for Analysis"))
display(Markdown(f"```python\n{vulnerable_code}\n```"))


### 🎯 Target Code for Analysis

```python
import sqlite3

def get_user_data(username: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    # VULNERABILITY: Unsanitized user input directly interpolated into SQL query
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    result = cursor.fetchall()
    conn.close()
    return result

```

### 2.1 🎮 Hacker Challenge: Which Input Unlocks the Whole Database?

Test your understanding with this interactive puzzle! Pick one of the inputs below to see how the database reacts in real time:

In [3]:
# Set up an in-memory database to test the puzzle live
test_db = sqlite3.connect(":memory:")
cur = test_db.cursor()
cur.execute("CREATE TABLE users (id INT, username TEXT, secret_coins INT)")
cur.executemany("INSERT INTO users VALUES (?, ?, ?)", [
    (1, 'alice', 150),
    (2, 'bob', 75),
    (3, 'admin', 999999)
])
test_db.commit()

sql_puzzle_options = [
    ("Option A: 'admin' (Normal guess - only gives 1 account)", "admin"),
    ("Option B: 'admin\' OR \'1\'=\'1' (The Math Riddle Trick - Injects True!)", "admin' OR '1'='1"),
    ("Option C: 'please_let_me_in_123' (Polite password guess)", "please_let_me_in_123")
]

puzzle_selector = widgets.RadioButtons(
    options=[opt[0] for opt in sql_puzzle_options],
    description="Select Input:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

puzzle_btn = widgets.Button(
    description="Test Input on Database",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="220px")
)

puzzle_output = widgets.Output()

def on_puzzle_clicked(b):
    with puzzle_output:
        clear_output()
        choice = puzzle_selector.value
        # Resolve payload
        payload = next(opt[1] for opt in sql_puzzle_options if opt[0] == choice)
        query = f"SELECT * FROM users WHERE username = '{payload}'"
        
        display(Markdown(f"**Database Query Executed:**\n```sql\n{query}\n```"))
        try:
            res = cur.execute(query).fetchall()
            display(Markdown(f"**Database Returned:** `{res}`"))
            
            if len(res) > 1:
                display(Markdown("🎉 **JACKPOT! (SQL Injection Success)**\n\nBecause `'1'='1'` is always true, the computer returned **ALL ACCOUNTS** including the secret admin with 999,999 coins!"))
            elif len(res) == 1:
                display(Markdown("🔒 **Normal Result:** You only got 1 account back. The computer was not tricked."))
            else:
                display(Markdown("❌ **Access Denied:** No user found by that name. The hack failed."))
        except Exception as err:
            display(Markdown(f"💥 Syntax Error: `{err}`"))

puzzle_btn.on_click(on_puzzle_clicked)

display(Markdown("### 🧩 Hacker Lab: The Magic Password Riddle"))
display(puzzle_selector)
display(puzzle_btn)
display(puzzle_output)


### 🧩 Hacker Lab: The Magic Password Riddle

RadioButtons(description='Select Input:', layout=Layout(width='650px'), options=("Option A: 'admin' (Normal gu…

Button(button_style='primary', description='Test Input on Database', icon='play', layout=Layout(width='220px')…

Output()

## 3. Enforcing Structured Output with Pydantic

In production DevSecOps pipelines, downstream automation tools (SAST/DAST analyzers, ticketing systems, pull request bots) cannot reliably parse conversational markdown.

We define a strict **Pydantic schema** (`RemediationReport`) and supply it via `response_schema` to enforce guaranteed JSON output.

In [4]:
class RemediationReport(BaseModel):
    cwe_id: str = Field(description="The formal CWE identifier, e.g. 'CWE-89'")
    vulnerability_name: str = Field(description="Standard vulnerability category name, e.g. 'SQL Injection'")
    severity: str = Field(description="Severity rating: Low, Medium, High, or Critical")
    risk_explanation: str = Field(description="One concise sentence explaining the exploitation risk.")
    vulnerable_lines: list[int] = Field(description="Line numbers in the original code snippet containing the flaw.")
    remediated_code: str = Field(description="Complete, syntactically valid remediated Python code using safe parameterized queries and context managers.")
    remediation_notes: str = Field(description="Summary of hardening best practices applied in the remediated code.")

display(Markdown("✅ **Pydantic Schema `RemediationReport` declared.**"))


✅ **Pydantic Schema `RemediationReport` declared.**

## 4. Single-Shot Vulnerability Discovery & Remediation Execution

We now execute the audit with the selected model. We wrap the API call in `tenacity` retry logic to ensure resilience against transient 500/503 errors, validate the structured response against `RemediationReport`, and render a rich DevSecOps advisory report.

In [5]:
# Resolve selected model from interactive dropdown (defaults to gemini-3.8-flash)
selected_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
display(Markdown(f"*Auditing code using model:* **`{selected_model}`**..."))

prompt = f"""You are a senior DevSecOps architect auditing source code in a CI/CD pipeline.
Analyze the following Python snippet for security vulnerabilities.
Identify the specific CWE, assess exploitation risk, and produce a hardened, production-ready patch.

Code to analyze:
{vulnerable_code}
"""

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=6),
    retry=retry_if_exception_type(ServerError),
    before_sleep=lambda r: display(Markdown(f"⚠️ *API busy (503/500). Retrying in {r.next_action.sleep:.1f}s...*"))
)
def _raw_audit_call(model_name: str, code_prompt: str) -> types.GenerateContentResponse:
    return client.models.generate_content(
        model=model_name,
        contents=code_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RemediationReport,
            temperature=0.1,  # Low temperature for deterministic analysis
        )
    )

def run_vulnerability_audit(model_name: str, code_prompt: str) -> types.GenerateContentResponse:
    try:
        return _raw_audit_call(model_name, code_prompt)
    except Exception as err:
        fallback = "gemini-2.5-flash"
        if model_name != fallback:
            display(Markdown(f"⚠️ *Model `{model_name}` temporarily unavailable. Automatically failing over to resilient `{fallback}`...*"))
            return _raw_audit_call(fallback, code_prompt)
        raise

def render_remediation_report(rep: RemediationReport):
    """Helper to render a visually rich DevSecOps security report card."""
    severity_colors = {
        "CRITICAL": "#d32f2f",
        "HIGH": "#e65100",
        "MEDIUM": "#f57c00",
        "LOW": "#388e3c"
    }
    color = severity_colors.get(rep.severity.upper(), "#1976d2")
    cwe_num = rep.cwe_id.replace("CWE-", "").strip()

    report_display = f"""
### 🛡️ DevSecOps Automated Vulnerability Report

| Attribute | Details |
| :--- | :--- |
| **Vulnerability** | **{rep.vulnerability_name}** |
| **CWE ID** | [`{rep.cwe_id}`](https://cwe.mitre.org/data/definitions/{cwe_num}.html) |
| **Severity** | <span style="color: {color}; font-weight: bold; font-size: 1.1em;">{rep.severity.upper()}</span> |
| **Vulnerable Lines** | `{rep.vulnerable_lines}` |

#### ⚠️ Exploitation Risk
> {rep.risk_explanation}

#### 🔧 Remediated Code
```python
{rep.remediated_code.strip()}
```

#### 📋 Remediation Notes
{rep.remediation_notes}
"""
    display(Markdown(report_display))

try:
    response = run_vulnerability_audit(selected_model, prompt)
    report = RemediationReport.model_validate_json(response.text)
    render_remediation_report(report)

except Exception as e:
    display(Markdown(f"❌ **Unexpected Error:** `{e}`"))


*Auditing code using model:* **`gemini-3.8-flash`**...

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



### 🛡️ DevSecOps Automated Vulnerability Report

| Attribute | Details |
| :--- | :--- |
| **Vulnerability** | **SQL Injection** |
| **CWE ID** | [`CWE-89`](https://cwe.mitre.org/data/definitions/89.html) |
| **Severity** | <span style="color: #d32f2f; font-weight: bold; font-size: 1.1em;">CRITICAL</span> |
| **Vulnerable Lines** | `[7, 8]` |

#### ⚠️ Exploitation Risk
> An attacker can inject arbitrary SQL commands via the username parameter to bypass authentication, expose confidential database records, or execute unauthorized database modifications.

#### 🔧 Remediated Code
```python
import sqlite3
from typing import List, Tuple, Any

def get_user_data(username: str) -> List[Tuple[Any, ...]]:
    query = "SELECT * FROM users WHERE username = ?"
    with sqlite3.connect('users.db') as conn:
        cursor = conn.cursor()
        cursor.execute(query, (username,))
        return cursor.fetchall()
```

#### 📋 Remediation Notes
Replaced dynamic string formatting (f-string) with parameterized query placeholders (?) to ensure database-level escaping, and implemented context manager handling for safe connection lifecycle management.


## 5. Visualizing the Remediation: Side-by-Side Patch Diff

> 💡 **How the Code Doctor Heals the Lock:**
> Look at the red `-` lines versus the green `+` lines in the diff below:
> * **Before (Broken):** `f"SELECT ... '{username}'"` (Glues user words into the sentence).
> * **After (Healed):** `SELECT ... ?`, `(username,)` (Puts user input inside an **unbreakable glass envelope** `?`). The database treats it strictly as safe text, never as commands!

In modern DevSecOps code review workflows, changes must be reviewed as precise line-by-line diffs. Below, we generate a unified syntax-highlighted diff comparing the insecure original function against the AI-generated remediated function.

In [6]:
def display_patch_diff(original: str, patched: str):
    """Generates and renders a syntax-highlighted unified diff between original and patched code."""
    orig_lines = original.strip().splitlines()
    patch_lines = patched.strip().splitlines()
    
    diff = list(difflib.unified_diff(
        orig_lines,
        patch_lines,
        fromfile="vulnerable_code.py (Insecure String Formatting)",
        tofile="remediated_code.py (Hardened Parameterization)",
        lineterm=""
    ))
    
    if diff:
        diff_text = "\n".join(diff)
        display(Markdown(f"### 🔍 Visual Remediation Diff\n```diff\n{diff_text}\n```"))
    else:
        display(Markdown("⚠️ *No differences detected between snippets.*\n"))

if 'report' in globals():
    display_patch_diff(vulnerable_code, report.remediated_code)
else:
    display(Markdown("⚠️ *Run Cell 4 to generate the report before visualizing the diff.*\n"))


### 🔍 Visual Remediation Diff
```diff
--- vulnerable_code.py (Insecure String Formatting)
+++ remediated_code.py (Hardened Parameterization)
@@ -1,11 +1,9 @@
 import sqlite3
+from typing import List, Tuple, Any
 
-def get_user_data(username: str):
-    conn = sqlite3.connect('users.db')
-    cursor = conn.cursor()
-    # VULNERABILITY: Unsanitized user input directly interpolated into SQL query
-    query = f"SELECT * FROM users WHERE username = '{username}'"
-    cursor.execute(query)
-    result = cursor.fetchall()
-    conn.close()
-    return result
+def get_user_data(username: str) -> List[Tuple[Any, ...]]:
+    query = "SELECT * FROM users WHERE username = ?"
+    with sqlite3.connect('users.db') as conn:
+        cursor = conn.cursor()
+        cursor.execute(query, (username,))
+        return cursor.fetchall()
```

## 6. Closed-Loop Automated Patch Verification (Autonomous Healing)

> 🧪 **The Blast Chamber Test:**
> Never blindly trust an AI patch! In our blast chamber, we test the hacker's trick on the old code (it fails), then re-test it on the new code (it blocks the attack), and finally make sure regular users can still log in without breaking anything!

A critical pitfall in AI-assisted code generation is blind trust: **did the synthesized patch actually neutralize the vulnerability without introducing regressions?**

In this section, we build an automated **Closed-Loop Verification Harness**:
1. **Setup Sandbox Database:** Populate a temporary SQLite database containing mock confidential user secrets.
2. **Execute Attack against Insecure Code:** Prove the SQL injection exploit (`admin' OR '1'='1`) succeeds against `vulnerable_code`, exfiltrating unauthorized records.
3. **Dynamic Compilation of AI Patch:** Compile `report.remediated_code` in an isolated execution namespace using `exec()`.
4. **Re-Test Exploit Payload:** Verify that the malicious payload is completely neutralized (returns 0 leaked records).
5. **Regression Testing:** Verify that legitimate lookups (`alice`) continue to function normally.

In [7]:
def setup_test_environment(db_path="users.db"):
    """Initializes a local SQLite database with test users and sensitive records."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute("DROP TABLE IF EXISTS users")
    c.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, secret TEXT)")
    c.execute("INSERT INTO users VALUES (1, 'alice', 'alice_confidential_token')")
    c.execute("INSERT INTO users VALUES (2, 'bob', 'bob_internal_key')")
    c.execute("INSERT INTO users VALUES (3, 'admin', 'MASTER_DATABASE_ADMIN_SECRET')")
    conn.commit()
    conn.close()

# Initialize sandbox database
setup_test_environment()

# 1. Test Insecure Baseline
vuln_ns = {"sqlite3": sqlite3}
exec(vulnerable_code, vuln_ns)
exploit_payload = "admin' OR '1'='1"
insecure_result = vuln_ns["get_user_data"](exploit_payload)

# 2. Test Remediated Patch Dynamically
remediated_ns = {"sqlite3": sqlite3}
if 'report' in globals():
    exec(report.remediated_code, remediated_ns)
    patched_exploit_result = remediated_ns["get_user_data"](exploit_payload)
    patched_legit_result = remediated_ns["get_user_data"]("alice")
    
    exploit_blocked = (len(patched_exploit_result) == 0)
    regression_passed = (len(patched_legit_result) == 1 and patched_legit_result[0][1] == "alice")
    
    verification_report = f"""
### 🧪 Closed-Loop Automated Patch Verification Certificate

| Verification Test | Target | Result | Status |
| :--- | :--- | :--- | :--- |
| **Baseline Exploit Attempt** | `vulnerable_code` | **{len(insecure_result)} records leaked** (Unchecked SQLi) | <span style="color:#d32f2f; font-weight:bold;">VULNERABLE ❌</span> |
| **Neutralization Test** | `remediated_code` | **{len(patched_exploit_result)} records returned** (Payload neutralized) | <span style="color:#388e3c; font-weight:bold;">{'BLOCKED ✅' if exploit_blocked else 'FAILED ❌'}</span> |
| **Functional Regression Test** | `remediated_code` | **Retrieved: `{patched_legit_result}`** | <span style="color:#388e3c; font-weight:bold;">{'PASSED ✅' if regression_passed else 'REGRESSION ❌'}</span> |

> **🎯 Remediation Verdict:** **{'AUTOMATED PATCH VERIFIED & SECURED ✅' if (exploit_blocked and regression_passed) else 'PATCH VERIFICATION FAILED ❌'}**
"""
    display(Markdown(verification_report))
else:
    display(Markdown("⚠️ *Run Cell 4 to generate the remediation report before executing verification.*\n"))



### 🧪 Closed-Loop Automated Patch Verification Certificate

| Verification Test | Target | Result | Status |
| :--- | :--- | :--- | :--- |
| **Baseline Exploit Attempt** | `vulnerable_code` | **3 records leaked** (Unchecked SQLi) | <span style="color:#d32f2f; font-weight:bold;">VULNERABLE ❌</span> |
| **Neutralization Test** | `remediated_code` | **0 records returned** (Payload neutralized) | <span style="color:#388e3c; font-weight:bold;">BLOCKED ✅</span> |
| **Functional Regression Test** | `remediated_code` | **Retrieved: `[(1, 'alice', 'alice_confidential_token')]`** | <span style="color:#388e3c; font-weight:bold;">PASSED ✅</span> |

> **🎯 Remediation Verdict:** **AUTOMATED PATCH VERIFIED & SECURED ✅**


## 7. Key DevSecOps Takeaways & Principles

1. **Parameterization over Sanitization:** Input sanitization (e.g. attempting to filter or escape quotes with regular expressions) is notoriously prone to bypasses. Parameterized queries (`?` placeholder) pass user data separately from the SQL instructions, preventing syntax corruption at the database protocol layer.
2. **Resource Lifecycle Management:** Using context managers (`with sqlite3.connect(...) as conn:`) prevents connection leaks and dangling transactions.
3. **Deterministic CI/CD Integration:** By forcing the model to adhere to a Pydantic schema (`response_schema=RemediationReport`), you can write automated policy gates into GitHub Actions or GitLab CI (e.g., automatically block merges if `report.severity in ['HIGH', 'CRITICAL']`).
4. **Autonomous Closed-Loop Verification:** Always pair AI patch synthesis with dynamic unit test execution to guarantee security without breaking existing functionality.

> **💡 Student Discussion Question:** *Why does escaping special characters (like quotes) fail in complex SQL dialects compared to true prepared statements?*

---


## 8. Multi-Class Vulnerability Suite (Expanding Audit Breadth)

Real-world DevSecOps security pipelines must defend against a broad taxonomy of weaknesses beyond SQL injection. Below, we define a curated suite covering four additional high-risk vulnerability classes:

1. **Path Traversal (`CWE-22`):** Direct file path concatenation allowing arbitrary local file read (`/etc/passwd`).
2. **OS Command Injection (`CWE-78`):** Unsanitized input passed to `shell=True` executing chained OS binaries.
3. **Server-Side Request Forgery (`CWE-918`):** Unchecked HTTP requests enabling probing of internal services and cloud metadata (`169.254.169.254`).
4. **Insecure Deserialization (`CWE-502`):** Untrusted `pickle` deserialization leading to arbitrary Remote Code Execution (RCE).

Use the interactive suite selector below to audit each flaw individually, or run the batch audit to compile a full **DevSecOps Compliance Matrix**.

In [8]:
# Curated Multi-Class Vulnerability Suite
VULNERABILITY_SUITE = {
    "Path Traversal (CWE-22)": """import os

def read_user_profile(filename: str):
    # VULNERABILITY: Direct concatenation allows path traversal (e.g. '../../etc/passwd')
    base_dir = '/var/data/profiles'
    filepath = os.path.join(base_dir, filename)
    with open(filepath, 'r') as f:
        return f.read()
""",
    "OS Command Injection (CWE-78)": """import subprocess

def ping_server(hostname: str):
    # VULNERABILITY: shell=True with unvalidated string allows command chaining (e.g. '8.8.8.8; whoami')
    cmd = f"ping -c 1 {hostname}"
    result = subprocess.check_output(cmd, shell=True)
    return result.decode()
""",
    "Server-Side Request Forgery (CWE-918)": """import requests

def fetch_remote_avatar(avatar_url: str):
    # VULNERABILITY: Unvalidated external URL allows SSRF against internal metadata (169.254.169.254)
    response = requests.get(avatar_url, timeout=5)
    return response.content
""",
    "Insecure Deserialization (CWE-502)": """import pickle
import base64

def restore_session(token_b64: str):
    # VULNERABILITY: Unpickling untrusted payload executes arbitrary code via __reduce__
    raw_bytes = base64.b64decode(token_b64)
    return pickle.loads(raw_bytes)
"""
}

suite_selector = widgets.Dropdown(
    options=list(VULNERABILITY_SUITE.keys()),
    value="Path Traversal (CWE-22)",
    description="Select Flaw:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

audit_target_btn = widgets.Button(
    description="Audit Selected Flaw",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="200px")
)

suite_output = widgets.Output()

def on_audit_target_clicked(b):
    with suite_output:
        clear_output()
        chosen_name = suite_selector.value
        target_snippet = VULNERABILITY_SUITE[chosen_name]
        target_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
        
        display(Markdown(f"### 🎯 Target Snippet: `{chosen_name}`"))
        display(Markdown(f"```python\n{target_snippet}\n```"))
        display(Markdown(f"*Auditing with model:* **`{target_model}`**..."))
        
        audit_prompt = f"""You are a senior DevSecOps architect. Analyze this code for vulnerabilities.
Identify the CWE, explain exploitation risk, and provide hardened, secure replacement code.

Code:
{target_snippet}
"""
        try:
            res = run_vulnerability_audit(target_model, audit_prompt)
            rep = RemediationReport.model_validate_json(res.text)
            render_remediation_report(rep)
            display_patch_diff(target_snippet, rep.remediated_code)
        except Exception as e:
            display(Markdown(f"❌ **Audit Error:** `{e}`"))

audit_target_btn.on_click(on_audit_target_clicked)

display(Markdown("### 🎛️ Interactive Vulnerability Audit Suite"))
display(widgets.HBox([suite_selector, audit_target_btn]))
display(suite_output)


### 🎛️ Interactive Vulnerability Audit Suite

Output()

### 8.1 Automated Batch Suite Audit & Compliance Matrix

Click the button below to execute an automated batch audit across the entire vulnerability suite. The pipeline validates each finding with Pydantic and aggregates the results into a consolidated **DevSecOps Compliance Matrix**.

In [9]:
batch_audit_btn = widgets.Button(
    description="Run Full Suite Batch Audit",
    button_style="success",
    icon="tasks",
    layout=widgets.Layout(width="260px")
)

batch_output = widgets.Output()

def on_batch_audit_clicked(b):
    with batch_output:
        clear_output()
        target_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
        display(Markdown(f"### 📊 Running Batch DevSecOps Audit on 4 Vulnerability Classes using `{target_model}`..."))
        
        audit_results = []
        for flaw_name, snippet in VULNERABILITY_SUITE.items():
            display(Markdown(f"Auditing **{flaw_name}**..."))
            prompt_text = f"""You are a senior DevSecOps architect.
Analyze the code, identify the CWE, explain risk in 1 sentence, and provide hardened replacement code.

Code:
{snippet}
"""
            try:
                resp = run_vulnerability_audit(target_model, prompt_text)
                parsed = RemediationReport.model_validate_json(resp.text)
                audit_results.append((flaw_name, parsed))
            except Exception as err:
                display(Markdown(f"⚠️ Failed on {flaw_name}: `{err}`"))
        
        # Build Consolidated Compliance Table
        table_rows = ["| Vulnerability Class | Detected CWE | Severity | Exploitation Risk Summary |", "|:---|:---|:---|:---|"]
        for name, r in audit_results:
            cwe_link = f"[{r.cwe_id}](https://cwe.mitre.org/data/definitions/{r.cwe_id.replace('CWE-', '')}.html)"
            table_rows.append(f"| **{name}** | {cwe_link} | **{r.severity.upper()}** | {r.risk_explanation} |")
        
        display(Markdown("### 📋 Consolidated DevSecOps Compliance Matrix"))
        display(Markdown("\n".join(table_rows)))

batch_audit_btn.on_click(on_batch_audit_clicked)

display(batch_audit_btn)
display(batch_output)


Button(button_style='success', description='Run Full Suite Batch Audit', icon='tasks', layout=Layout(width='26…

Output()

---
